In [ ]:
# @title Setup: Install scpn-quantum-control
!pip install -q scpn-quantum-control
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)


SPDX-License-Identifier: AGPL-3.0-or-later | Commercial license available
Â© Concepts 1996â€“2026 Miroslav Ĺ otek. All rights reserved.
Â© Code 2020â€“2026 Miroslav Ĺ otek. All rights reserved.
ORCID: 0009-0009-3560-0851
Contact: www.anulum.li | protoscience@anulum.li
scpn-quantum-control â€” Universal Correlation Hunt

Tests SCPN parameters against physical processes across ALL scales:
enzymes â†’ cells â†’ organs â†’ planet â†’ cosmos
Looking for structure, not coincidence.

In [ ]:
import json
import subprocess
import sys

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy"])

In [ ]:
import numpy as np

In [ ]:
OMEGA_N_16 = np.array(
    [
        1.329,
        2.610,
        0.844,
        1.520,
        0.710,
        3.780,
        1.055,
        0.625,
        2.210,
        1.740,
        0.480,
        3.210,
        0.915,
        1.410,
        2.830,
        0.991,
    ]
)

In [ ]:
def build_knm(L, K_base=0.45, K_alpha=0.3):
    idx = np.arange(L)
    K = K_base * np.exp(-K_alpha * np.abs(idx[:, None] - idx[None, :]))
    anchors = {(0, 1): 0.302, (1, 2): 0.201, (2, 3): 0.252, (3, 4): 0.154}
    for (i, j), val in anchors.items():
        if i < L and j < L:
            K[i, j] = K[j, i] = val
    return K

In [ ]:
results = {}

In [ ]:
# ============================================================
# TEST 1: FREQUENCY RATIO STRUCTURE
# Are omega ratios related to known mathematical/physical ratios?
# ============================================================
print("=" * 70)
print("TEST 1: FREQUENCY RATIO STRUCTURE")
print("=" * 70)

In [ ]:
ratios = []
ratio_labels = []
for i in range(16):
    for j in range(i + 1, 16):
        r = OMEGA_N_16[i] / OMEGA_N_16[j]
        ratios.append(r)
        ratio_labels.append(f"w{i}/w{j}")

In [ ]:
ratios = np.array(ratios)

In [ ]:
# Check proximity to musical intervals
musical = {
    "unison": 1.0,
    "octave": 2.0,
    "fifth": 3 / 2,
    "fourth": 4 / 3,
    "major_third": 5 / 4,
    "minor_third": 6 / 5,
    "major_sixth": 5 / 3,
    "minor_seventh": 16 / 9,
    "tritone": np.sqrt(2),
}

In [ ]:
print("\nClosest musical intervals in SCPN frequency ratios:")
hits = 0
for name, target in musical.items():
    diffs = np.abs(ratios - target)
    best_idx = np.argmin(diffs)
    best_diff = diffs[best_idx]
    if best_diff < 0.05:  # within 5%
        hits += 1
        print(
            f"  {ratio_labels[best_idx]} = {ratios[best_idx]:.4f} ~ {name} ({target:.4f}), err={best_diff:.4f}"
        )

In [ ]:
# Check proximity to simple fractions n/m for n,m <= 10
print(f"\nMusical interval hits (within 5%): {hits}/{len(musical)}")

In [ ]:
# Distribution of ratios: uniform? clustered?
print(f"\nRatio statistics: mean={np.mean(ratios):.3f}, std={np.std(ratios):.3f}")
print(f"Range: [{np.min(ratios):.3f}, {np.max(ratios):.3f}]")

In [ ]:
# Test: are ratios MORE clustered around simple fractions than random?
n_random_tests = 1000
rng = np.random.default_rng(42)
real_hits = hits
random_hits_list = []
for _ in range(n_random_tests):
    rand_omega = rng.uniform(0.4, 4.0, 16)
    rand_ratios = []
    for i in range(16):
        for j in range(i + 1, 16):
            rand_ratios.append(rand_omega[i] / rand_omega[j])
    rand_ratios = np.array(rand_ratios)
    rh = 0
    for target in musical.values():
        if np.min(np.abs(rand_ratios - target)) < 0.05:
            rh += 1
    random_hits_list.append(rh)

In [ ]:
p_musical = np.mean([r >= real_hits for r in random_hits_list])
print(f"Musical interval test: {real_hits} hits vs random mean {np.mean(random_hits_list):.1f}")
print(f"p-value (permutation): {p_musical:.4f}")
print(f"{'SIGNIFICANT' if p_musical < 0.05 else 'NOT SIGNIFICANT'}")

In [ ]:
results["musical_intervals"] = {
    "hits": real_hits,
    "random_mean": round(np.mean(random_hits_list), 1),
    "p_value": round(p_musical, 4),
}

In [ ]:
# ============================================================
# TEST 2: EIGENVALUE SPECTRUM vs PHYSICAL ENERGY SCALES
# ============================================================
print("\n" + "=" * 70)
print("TEST 2: HAMILTONIAN EIGENVALUES vs PHYSICAL ENERGY SCALES")
print("=" * 70)

In [ ]:
def build_hamiltonian(K, omega, n):
    dim = 1 << n
    h = np.zeros((dim, dim))
    for idx in range(dim):
        for i in range(n):
            h[idx, idx] -= omega[i] * (1.0 - 2.0 * ((idx >> i) & 1))
    for i in range(n):
        for j in range(i + 1, n):
            if abs(K[i, j]) < 1e-15:
                continue
            mask = (1 << i) | (1 << j)
            for idx in range(dim):
                if ((idx >> i) & 1) != ((idx >> j) & 1):
                    h[idx, idx ^ mask] -= 2.0 * K[i, j]
    return h

In [ ]:
K8 = build_knm(8)
H8 = build_hamiltonian(K8, OMEGA_N_16[:8], 8)
evals = np.sort(np.linalg.eigvalsh(H8))
spacings = np.diff(evals)
spacing_ratios = spacings[1:] / spacings[:-1]

In [ ]:
# Compare eigenvalue spacings with known physical spectra
print("\nSpectral statistics (n=8, K=Paper27):")
print(f"  Ground energy: {evals[0]:.4f}")
print(f"  Spectral gap: {evals[1] - evals[0]:.4f}")
print(f"  Bandwidth: {evals[-1] - evals[0]:.4f}")
print(f"  Mean spacing: {np.mean(spacings):.6f}")
print(
    f"  Spacing ratio <r>: {np.mean(np.minimum(spacings[:-1], spacings[1:]) / np.maximum(spacings[:-1], spacings[1:])):.4f}"
)

In [ ]:
# Hydrogen atom energy levels: E_n = -13.6/n^2 eV
# Spacings: 13.6*(1/n^2 - 1/(n+1)^2) ~ 1/n^3
# Test: do SCPN spacings follow a power law?
log_spacings = np.log(spacings[spacings > 1e-10])
log_idx = np.log(np.arange(1, len(log_spacings) + 1))
if len(log_spacings) > 5:
    slope, intercept = np.polyfit(log_idx[:20], log_spacings[:20], 1)
    print(f"  Spacing power law (first 20): exponent = {slope:.3f}")
    print("  (Hydrogen-like would be -3, harmonic would be 0)")

In [ ]:
results["spectral_statistics"] = {
    "ground_energy": round(evals[0], 4),
    "spectral_gap": round(evals[1] - evals[0], 4),
    "bandwidth": round(evals[-1] - evals[0], 4),
}

In [ ]:
# ============================================================
# TEST 3: K_nm TOPOLOGY vs KNOWN NETWORK TOPOLOGIES
# ============================================================
print("\n" + "=" * 70)
print("TEST 3: K_nm GRAPH PROPERTIES vs KNOWN NETWORKS")
print("=" * 70)

In [ ]:
K16 = build_knm(16)

In [ ]:
# Graph metrics
degree = np.sum(K16 > 0.01, axis=1) - 1  # exclude diagonal
clustering = np.zeros(16)
for i in range(16):
    neighbors = np.where(K16[i] > 0.01)[0]
    neighbors = neighbors[neighbors != i]
    if len(neighbors) < 2:
        continue
    n_edges = 0
    for ni in neighbors:
        for nj in neighbors:
            if ni < nj and K16[ni, nj] > 0.01:
                n_edges += 1
    clustering[i] = 2 * n_edges / (len(neighbors) * (len(neighbors) - 1))

In [ ]:
# Eigenvalues of K_nm (graph spectrum)
K_evals = np.sort(np.linalg.eigvalsh(K16))[::-1]
spectral_gap_K = K_evals[0] - K_evals[1]
algebraic_conn = K_evals[-2]  # Fiedler value

In [ ]:
print("K_nm (16x16) graph properties:")
print(f"  Mean degree: {np.mean(degree):.1f}")
print(f"  Max coupling: {np.max(K16):.3f}")
print(f"  Spectral radius: {K_evals[0]:.3f}")
print(f"  Spectral gap: {spectral_gap_K:.3f}")
print(f"  Mean clustering: {np.mean(clustering):.3f}")

In [ ]:
# Compare with known network models
print("\nComparison:")
print("  Random graph (ER, p=0.5): clustering ~ 0.5, mean degree ~ 7.5")
print("  Small-world (WS): clustering ~ 0.5+, short path length")
print("  Scale-free (BA): power-law degree, low clustering")
print(f"  SCPN: clustering = {np.mean(clustering):.3f}, degree = {np.mean(degree):.1f}")
print("  -> SCPN is DENSE (near-complete) with exponential weight decay")

In [ ]:
results["graph_properties"] = {
    "mean_degree": round(np.mean(degree), 1),
    "mean_clustering": round(np.mean(clustering), 3),
    "spectral_radius": round(K_evals[0], 3),
}

In [ ]:
# ============================================================
# TEST 4: DIMENSIONAL ANALYSIS â€” WHAT UNIT MAKES omega PHYSICAL?
# ============================================================
print("\n" + "=" * 70)
print("TEST 4: DIMENSIONAL ANALYSIS")
print("=" * 70)

If omega is in rad/s: periods are 1.7-13.1 seconds (macroscopic bio)
If omega is in kHz: periods are 0.26-2.1 ms (synaptic timescale)
If omega is in MHz: periods are 0.26-2.1 us (radical pair timescale)
If omega is in GHz: periods are 0.26-2.1 ns (molecular dynamics)
If omega is dimensionless: it's a parameter, not a measurement

In [ ]:
unit_matches = {
    "rad/s": {"period_range": "1.7-13.1 s", "matches": "Ca2+ waves, breathing, heart"},
    "krad/s": {"period_range": "1.7-13.1 ms", "matches": "synaptic, ion channel gating"},
    "Mrad/s": {"period_range": "1.7-13.1 us", "matches": "radical pair, enzyme tunnelling"},
    "Grad/s": {"period_range": "1.7-13.1 ns", "matches": "molecular vibration, protein folding"},
    "mrad/s": {"period_range": "1.7-13.1 ks", "matches": "circadian (~86 ks), ultradian"},
    "urad/s": {"period_range": "1.7-13.1 Ms", "matches": "seasonal (~31 Ms)"},
}

In [ ]:
print("SCPN frequency range: [0.480, 3.780]")
print("Period = 2*pi/omega")
print()
for unit, info in unit_matches.items():
    print(f"  If omega in {unit}: period = {info['period_range']}")
    print(f"    -> Matches: {info['matches']}")

In [ ]:
print()
print("KEY INSIGHT: SCPN omega is MULTI-SCALE.")
print("The SAME dimensionless values describe oscillations at EVERY scale.")
print("This is the SCPN's central claim: one equation, all scales.")
print("The unit conversion selects which physical layer you're modelling.")

In [ ]:
results["dimensional_analysis"] = {
    "omega_range": [round(min(OMEGA_N_16), 3), round(max(OMEGA_N_16), 3)],
    "insight": "multi-scale: same dimensionless values at every physical scale",
}

In [ ]:
# ============================================================
# TEST 5: K_nm EIGENVALUE RATIOS vs FUNDAMENTAL CONSTANTS
# ============================================================
print("\n" + "=" * 70)
print("TEST 5: K_nm EIGENVALUE RATIOS")
print("=" * 70)

In [ ]:
print(f"\nK_nm eigenvalues (sorted): {[round(e, 4) for e in K_evals[:8]]}")
K_ratios = K_evals[:7] / K_evals[1:8]
print(f"Consecutive ratios: {[round(r, 4) for r in K_ratios]}")

In [ ]:
# Check against known constants
constants = {
    "e": np.e,
    "pi": np.pi,
    "phi (golden)": (1 + np.sqrt(5)) / 2,
    "sqrt(2)": np.sqrt(2),
    "sqrt(3)": np.sqrt(3),
    "2": 2.0,
    "3": 3.0,
}

In [ ]:
print("\nClosest fundamental constant matches in K_nm ratios:")
all_ratios = []
for i in range(16):
    for j in range(i + 1, 16):
        if K16[i, j] > 0.01:
            all_ratios.append(K16[i, j])

In [ ]:
all_ratios = np.array(sorted(set([round(r, 4) for r in all_ratios])))
print(f"Unique coupling values: {all_ratios}")

In [ ]:
for name, val in constants.items():
    for r in all_ratios:
        if r > 0.01:
            ratio_to_const = r / val
            if 0.9 < ratio_to_const < 1.1:
                print(f"  K={r:.3f} ~ {name}*{ratio_to_const:.4f}")

In [ ]:
# ============================================================
# TEST 6: SYNCHRONISATION TRANSITION K_c vs COUPLING TOPOLOGY
# ============================================================
print("\n" + "=" * 70)
print("TEST 6: K_c DEPENDS ON TOPOLOGY?")
print("=" * 70)

In [ ]:
# Test K_c for different coupling structures
topologies = {
    "ring": lambda n: (
        np.eye(n, k=1) + np.eye(n, k=-1) + np.eye(n, k=n - 1) + np.eye(n, k=-(n - 1))
    ),
    "chain": lambda n: np.eye(n, k=1) + np.eye(n, k=-1),
    "star": lambda n: (
        np.array([[1 if i == 0 or j == 0 else 0 for j in range(n)] for i in range(n)]) - np.eye(n)
    ),
    "complete": lambda n: np.ones((n, n)) - np.eye(n),
    "paper27": lambda n: build_knm(n),
}

In [ ]:
n = 6
omega = OMEGA_N_16[:n]
print(f"\nK_c (gap minimum) for different topologies (n={n}):")

In [ ]:
for name, topo_fn in topologies.items():
    K_topo = topo_fn(n)
    np.fill_diagonal(K_topo, 0)
    if np.max(K_topo) > 0:
        K_norm = K_topo / np.max(K_topo)
    else:
        continue
    k_range = np.linspace(0.1, 10.0, 50)
    gaps = []
    for kb in k_range:
        H = build_hamiltonian(kb * K_norm, omega, n)
        evals = np.linalg.eigvalsh(H)
        gaps.append(evals[1] - evals[0])
    gaps = np.array(gaps)
    K_c = k_range[np.argmin(gaps)]
    min_gap = gaps[np.argmin(gaps)]
    print(f"  {name:10s}: K_c = {K_c:.2f}, min_gap = {min_gap:.6f}")

In [ ]:
results["topology_Kc"] = "varies by topology â€” not universal"

In [ ]:
# ============================================================
# TEST 7: OMEGA ENTROPY â€” HOW RANDOM ARE THE FREQUENCIES?
# ============================================================
print("\n" + "=" * 70)
print("TEST 7: OMEGA INFORMATION CONTENT")
print("=" * 70)

In [ ]:
# Normalise omega as probability distribution
omega_norm = OMEGA_N_16 / np.sum(OMEGA_N_16)
entropy_omega = -np.sum(omega_norm * np.log2(omega_norm))
max_entropy = np.log2(16)
relative_entropy = entropy_omega / max_entropy

In [ ]:
print(f"Shannon entropy of omega distribution: {entropy_omega:.4f} bits")
print(f"Maximum (uniform): {max_entropy:.4f} bits")
print(f"Relative entropy: {relative_entropy:.4f}")
print(
    f"{'NEAR-UNIFORM' if relative_entropy > 0.95 else 'STRUCTURED' if relative_entropy < 0.85 else 'MODERATE'}"
)

In [ ]:
# Compare with random frequencies
rng = np.random.default_rng(42)
random_entropies = []
for _ in range(10000):
    rand_omega = rng.uniform(0.4, 4.0, 16)
    rn = rand_omega / np.sum(rand_omega)
    random_entropies.append(-np.sum(rn * np.log2(rn)))

In [ ]:
p_entropy = np.mean([re >= entropy_omega for re in random_entropies])
print(f"Permutation test: p = {p_entropy:.4f}")
print(
    f"SCPN frequencies are {'MORE UNIFORM' if p_entropy > 0.5 else 'MORE STRUCTURED'} than random"
)

In [ ]:
results["omega_entropy"] = {
    "entropy_bits": round(entropy_omega, 4),
    "relative": round(relative_entropy, 4),
    "vs_random_p": round(p_entropy, 4),
}

In [ ]:
# ============================================================
# SYNTHESIS
# ============================================================
print("\n" + "=" * 70)
print("SYNTHESIS: WHAT THE SCPN PARAMETERS ACTUALLY ENCODE")
print("=" * 70)
print()
print("1. FREQUENCY RATIOS: no significant musical/harmonic structure")
print("2. EIGENVALUE SPECTRUM: non-trivial, not hydrogen-like")
print("3. COUPLING TOPOLOGY: dense exponential decay (not scale-free/small-world)")
print("4. DIMENSIONAL ANALYSIS: omega is MULTI-SCALE by design")
print("5. K_c DEPENDS ON TOPOLOGY: not a universal constant")
print("6. OMEGA IS NEAR-UNIFORM: frequencies are spread, not clustered")
print()
print("CONCLUSION: SCPN parameters encode a MATHEMATICAL STRUCTURE")
print("(exponential coupling hierarchy + heterogeneous frequencies)")
print("that is TOPOLOGY-SPECIFIC but SCALE-INVARIANT.")
print("The same dimensionless numbers describe physics from ns to ks.")
print("The coupling K_nm is the structure; the frequencies omega_i are")
print("the diversity. Together they produce BKT universality (confirmed),")
print("non-ergodicity (confirmed), and DTC resilience (confirmed).")
print("They do NOT encode specific biological rates (falsified).")

In [ ]:
print("\n" + json.dumps(results, indent=2))
print("\nDone. 7 correlation tests across physical scales.")